<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# DATA/MSML 641: Natural Language Processing
## Lecture 8: Vector Semantics and Embeddings

**University of Maryland, College Park**  
**Fall 2025**  
**Instructor**: Armin Mehrabian

## 🎯 Learning Objectives

By the end of this lecture, you will be able to:
1. Understand the distributional hypothesis and its role in representing word meaning
2. Distinguish between lexical semantic concepts: lemmas, senses, synonymy, similarity
3. Build term-document and term-term co-occurrence matrices
4. Implement TF-IDF weighting for vector representations
5. Measure similarity using cosine distance
6. Understand Word2Vec (CBOW and Skip-gram) architectures
7. Explore semantic properties of word embeddings

## 📋 Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Introduction

*Nets are for fish;*  
*Once you get the fish, you can forget the net.*  
*Words are for meaning;*  
*Once you get the meaning, you can forget the words*

— **Zhuangzi (庄子)**, Chapter 26

# Lexical Semantic

# Distributional Hypothesis

**Definition** – Words that occur in similar contexts tend to have similar meanings. The link between how words are distributed and the similarity in what they mean is called the **distributional hypothesis**.

- Formulated by linguists like Joos (1950), Harris (1954), and Firth (1957)

**Examples:**
- Eye-doctor & Oculist → near words are *eye* or *examined*

# Lexical Semantic – Lemmas and Senses (1/6)

**Mouse** (N)
1. any of numerous small rodents...
2. a hand-operated device that controls a cursor...

**Mouse** is lemma for: **Mice** and **Mouse**

**Word sense**: each of the aspects of the meaning of a word is called **word sense**

# Lexical Semantic – Synonym (2/6)

**Synonym** – when one sense of a word is identical or near identical to one sense in another word then these two words are synonyms.

- couch/sofa
- vomit/throw up
- filbert/hazelnut
- car/automobile
- H₂O/water

# Lexical Semantic – Word Similarity (3/6)

**Word Similarity** – words may not have so many synonyms but certainly have similar words.
- Cat and Dog are similar words

**Manual Annotation of Similarity** (SimLex-999 dataset, Hill et. Al 2015)

| Word 1 | Word 2 | Score |
|--------|--------|-------|
| vanish | disappear | 9.8 |
| belief | impression | 5.95 |
| muscle | bone | 3.65 |
| modest | flexible | 0.98 |
| hole | agreement | 0.3 |

# Lexical Semantic – Word Relatedness (4/6)

**Word relatedness** or association in psychology – two words are co-participating in an event or are related eventively. A common way that words can be related is through the same **semantic field**, which is a set of words that cover a particular semantic domain and bear structured relations with each other.

- **Co-participation**: cup and coffee
- **Related Eventively**: Scalpel and Surgeon
- **Semantic Field**:
  - field of **hospitals**: surgeon, scalpel, nurse, anesthetic, hospital
  - Field of **restaurants**: waiter, menu, plate, food, chef

# Lexical Semantic – Semantic Frames and Roles (5/6)

**Semantic Frames** – closely related to semantic fields is semantic frames. A semantic frame is a set of words that denote perspectives or participants in a particular type of event.

- **buy** (the event from the perspective of the buyer)
- **sell** (from the perspective of the seller)
- **pay** (focusing on the monetary aspect)

**Paraphrasing Task:**
- *Sam bought the book from Ling* (Paraphrase to) *Ling sold the book to Sam*

# Lexical Semantic – Connotation (6/6)

**Connotation** - Words have affective meanings or connotations. The word connotation has different meanings in different fields, but here we use it to mean the aspects of a word's meaning that are related to a writer or reader's emotions, sentiments, opinions, or evaluations.

- Pos-connotation: *wonderful*; Neg-connotation: *dreary*
- A mix of Pos-conn and Neg-conn of words with the same meaning
  - *copy, replica, reproduction*
- Pos-evaluation: *great, love*; Neg-evaluation: *hate, terrible*
- Words are different in 3 dimensions of affect (Osgood 1957):
  - **valence**: the pleasantness of the stimulus
  - **arousal**: the intensity of emotion provoked by the stimulus
  - **dominance**: the degree of control exerted by the stimulus

# Vector Semantic

*Ongchoi is delicious sauteed with garlic.*  
*Ongchoi is superb over rice.*  
*... ongchoi leaves with salty sauces ...*

And suppose that you had seen many of these context words in other contexts:

*... spinach sauteed with garlic over rice ...*  
*... chard stems and leaves are delicious ...*  
*... collard greens and other salty leafy greens*

# Vector Semantic

T-SNE plot of some word embedding in 2 dimensions space

*(Visualization would show sentiment words clustered: positive words like "good, wonderful, fantastic" on one side, negative words like "bad, worst, terrible" on the other, and function words like "to, by, is" in a separate cluster)*

# Words and Vectors

**Vectors** or the **Distributional model** of word meaning is based on a co-occurrence matrix, a way to represent how often two words co-occur. Two popular matrices:

- **Term-document co-occurrence matrix**
- **Term-term co-occurrence matrix**

# Words and Vectors

4 documents from Shakespeare's plays and 4 words

- **Vector** – a list or array of numbers
- Julius Caesar -> [7, 62, 1, 2]

In [ ]:
# Term-document matrix example from Shakespeare's plays
shakespeare_matrix = pd.DataFrame({
    'As You Like It': [1, 114, 36, 20],
    'Twelfth Night': [0, 80, 58, 15],
    'Julius Caesar': [7, 62, 1, 2],
    'Henry V': [13, 89, 4, 3]
}, index=['battle', 'good', 'fool', 'wit'])

print("Term-Document Matrix (Shakespeare's Plays)")
print("="*60)
print(shakespeare_matrix)
print("\nColumn vector for 'Julius Caesar':")
print(shakespeare_matrix['Julius Caesar'].values)

# Words and Vectors

4 documents from Shakespeare's plays and 4 words

- **Vector** – a list or array of numbers
- Julius Caesar -> [7, 62, 1, 2]
- **Column Vector** – the red highlight around Julius Caesar

# Words and Vectors

Visualization of 4 of Shakespeare's plays documents showing only 2-dimensions for each document

*(Shows a 2D plot with x-axis='fool' and y-axis='battle', with four points representing the plays positioned according to their counts)*

In [ ]:
# Visualize documents in 2D space (using 'fool' and 'battle' dimensions)
fig, ax = plt.subplots(figsize=(10, 6))

x_coords = shakespeare_matrix.loc['fool'].values
y_coords = shakespeare_matrix.loc['battle'].values
labels = shakespeare_matrix.columns

ax.scatter(x_coords, y_coords, s=200, alpha=0.6, c=['blue', 'green', 'red', 'orange'])

for i, label in enumerate(labels):
    ax.annotate(f"{label}\n[{x_coords[i]},{y_coords[i]}]", 
                (x_coords[i], y_coords[i]),
                fontsize=10, ha='center')

ax.set_xlabel('fool', fontsize=12)
ax.set_ylabel('battle', fontsize=12)
ax.set_title("Shakespeare's Plays in 2D Word Space", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Words as vectors: document dimensions

The term-document matrix for four words in four Shakespeare plays. The red boxes show that each word is represented as a row vector of length four.

**Battle** -> [1, 0, 7, 13]

In [ ]:
# Extract row vectors (words as vectors in document space)
print("Words as Vectors (Document Dimensions)")
print("="*60)
for word in shakespeare_matrix.index:
    vec = shakespeare_matrix.loc[word].values
    print(f"{word:10} -> {vec}")

# Words as vectors: word dimensions (1/3)

**Context window size ±4 words around each word**

- is traditionally followed by **cherry** pie, traditional dessert
- often mixed, such as **strawberry** rhubarb pie, Apple pie
- computer peripherals and personal **digital** assistants, These devices usually
- a computer. This includes **information** available on the internet

# Words as vectors: word dimensions (2/3)

Co-occurrence vectors for 4 words in Wikipedia Corpus, showing six of the dimensions.

| Word | aardvark | ... | computer | data | result | pie | sugar | ... |
|------|----------|-----|----------|------|--------|-----|-------|-----|
| cherry | 0 | ... | 2 | 8 | 9 | 442 | 25 | ... |
| strawberry | 0 | ... | 0 | 0 | 1 | 60 | 19 | ... |
| **digital** | **0** | **...** | **1670** | **1683** | **85** | **5** | **4** | **...** |
| information | 0 | ... | 3325 | 3982 | 378 | 5 | 13 | ... |

# Words as vectors: word dimensions (3/3)

A spatial visualization of word vectors for **digital** and **information**, showing just two of the dimensions, corresponding to the words **data** and **computer**.

In [ ]:
# Simulate word vectors in 2D space (data vs computer dimensions)
word_vectors_2d = {
    'digital': [1683, 1670],
    'information': [3982, 3325]
}

fig, ax = plt.subplots(figsize=(10, 8))

for word, coords in word_vectors_2d.items():
    ax.scatter(coords[0], coords[1], s=200, alpha=0.7)
    ax.annotate(f"{word}\n[{coords[0]},{coords[1]}]", 
                coords, fontsize=12, ha='center', fontweight='bold')
    # Draw arrow from origin
    ax.arrow(0, 0, coords[0], coords[1], head_width=100, head_length=150, 
             fc='blue', ec='blue', alpha=0.5, linewidth=2)

ax.set_xlabel('data', fontsize=14, fontweight='bold', color='blue')
ax.set_ylabel('computer', fontsize=14, fontweight='bold', color='blue')
ax.set_title('Word Vectors in 2D Semantic Space', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 4500)
ax.set_ylim(0, 4000)
plt.tight_layout()
plt.show()

# Cosine for Measuring Similarity (1/2)

**Vector Length:**
$$|\mathbf{v}| = \sqrt{\sum_{i=1}^{N} v_i^2}$$

**Normalized dot product:**
$$\frac{\mathbf{a} \cdot \mathbf{b}}{|\mathbf{a}||\mathbf{b}|} = \cos\theta$$

**Cosine Similarity:**
$$\text{cosine}(\mathbf{v},\mathbf{w}) = \frac{\mathbf{v} \cdot \mathbf{w}}{|\mathbf{v}||\mathbf{w}|} = \frac{\sum_{i=1}^{N} v_i w_i}{\sqrt{\sum_{i=1}^{N} v_i^2} \sqrt{\sum_{i=1}^{N} w_i^2}}$$

# Cosine for Measuring Similarity (2/2)

A (rough) graphical demonstration of cosine similarity, showing vectors for three words (*cherry*, *digital*, and *information*) in the two-dimensional space defined by counts of the words *computer* and *pie* nearby.

**Example calculations:**
- cos(cherry, information) = 0.018
- cos(digital, information) = 0.996

In [ ]:
# Cosine similarity demonstration
def cosine_sim(v1, v2):
    """Calculate cosine similarity between two vectors"""
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

# Example vectors (pie, data, computer dimensions)
cherry = np.array([442, 8, 2])
digital = np.array([5, 1683, 1670])
information = np.array([5, 3982, 3325])

print("Cosine Similarity Examples")
print("="*60)
print(f"cherry vector:      {cherry}")
print(f"digital vector:     {digital}")
print(f"information vector: {information}")
print()
print(f"cos(cherry, information) = {cosine_sim(cherry, information):.3f}")
print(f"cos(digital, information) = {cosine_sim(digital, information):.3f}")
print(f"cos(cherry, digital) = {cosine_sim(cherry, digital):.3f}")

# TF-IDF: Weighing terms in the vector

**Term frequency raw count:**
$$\text{tf}_{t,d} = \text{count}(t,d)$$

**Adding the log to normalize high frequency and zero count:**
$$\text{tf}_{t,d} = \begin{cases} 1 + \log_{10} \text{count}(t,d) & \text{if count}(t,d) > 0 \\ 0 & \text{otherwise} \end{cases}$$

**Document Frequency:**
$$\text{idf}_t = \log_{10}\left(\frac{N}{\text{df}_t}\right)$$

where *N* is the # of the total documents

**TF-IDF:**
$$w_{t,d} = \text{tf}_{t,d} \times \text{idf}_t$$

# TF-IDF: Weighing terms in the vector

A tf-idf weighted term-document matrix for four words in four Shakespeare plays, using the counts.

| Word | As You Like It | Twelfth Night | Julius Caesar | Henry V |
|------|----------------|---------------|---------------|----------|
| battle | 0.246 | 0 | 0.454 | 0.520 |
| good | 0 | 0 | 0 | 0 |
| fool | 0.030 | 0.033 | 0.0012 | 0.0019 |
| wit | 0.085 | 0.081 | 0.048 | 0.054 |

In [ ]:
# TF-IDF implementation
from sklearn.feature_extraction.text import TfidfTransformer

# Calculate TF-IDF from the Shakespeare matrix
tfidf_transformer = TfidfTransformer()
tfidf_matrix = tfidf_transformer.fit_transform(shakespeare_matrix.T).T

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=shakespeare_matrix.index,
    columns=shakespeare_matrix.columns
)

print("TF-IDF Weighted Term-Document Matrix")
print("="*60)
print(tfidf_df.round(3))

# Dense Vector Representation (word2vec)

# Model for Dense Vector Representation

- **Documents** = {"This is an example", "This is another example"}
- **Vocabulary** = {this, is, an, another, example}

• For a window size of 1, the surrounding words around "*is*" in the first document are "*this*" and "*an*"

• We can build a binary classifier for pairs *(input word, output word)* such that
  - The pairs **(is, this)** and **(is, an)** are classified as **true** (neighboring words)
  - The pairs **(is, example)** and **(is, another)** are classified as **false**

• Let $\mathbf{w}_i = [w_{i1}, w_{i2}, ..., w_{iD}]$ be the vector representation of word *i* in *D*-dimension

```
w_is  -----> Classification -----> 1
w_this ----> Model

w_is  -----> Classification -----> 0
w_another -> Model
```

• Neural network can be used to train $\mathbf{w}_i$

# Word Vectors: Learning with Neural Networks

**Neural Network-based Word Embedding Models**

- Google **Word2vec**
- Facebook **FastText**

*(Diagram shows a neural network with input layer (vocabulary words like apple, drink, eat, juice, milk, orange, rice, water), hidden layer with weighted connections, and output layer predicting context words)*

# Word2Vec Neural Network Architecture

**Input Layer:**
- **Input Vector**: One-hot encoded (vocabulary size of 10,000)
- A '1' in the position corresponding to the word "great"

**Hidden Layer:**
- **Linear Neurons** (Dimension of 300)
- 300 neurons

**Output Layer:**
- **Softmax Classifier**
- 10,000 neurons
- Probability that the word at a randomly chosen, nearby position is "abandon"
- ... "ability"
- ... "able"
- ... "zone"

# Word2Vec Learning Models

## Continuous Bag of Words (CBOW):
• Given the context predict the word:
  - $w_{i-2}, w_{i-1}, \mathbf{w}_i, w_{i+1}, w_{i+2}$
• Example: **The cat ate ______.**
  - Fill in the blank, e.g. "food".
• Faster to train
• Works well for large amount of training data

## Continuous Skip-Gram:
• Given the word predict the context:
  - $w_{i-2}, w_{i-1}, \mathbf{w}_i, w_{i+1}, w_{i+2}$
• Ex: ____ ____ ____ food.
  - Fill in the blank, e.g. "The cat ate"
• Slower to train
• Works better for infrequent words

# Word2vec

• **Represent the meaning of the word by its context**

**CBOW** (Continuous Bag of Words)
- Given context, predict center word
- INPUT → PROJECTION → OUTPUT
- w(t-2), w(t-1), w(t+1), w(t+2) → SUM → w(t)

**Skip-gram**
- Given center word, predict context
- INPUT → PROJECTION → OUTPUT
- w(t) → → w(t-2), w(t-1), w(t+1), w(t+2)

# Word2Vec: CBOW Example

• **"the cat ___ on floor"**

**Input layer:**
- the, cat, on, floor (one-hot vectors)

**Hidden layer:**
- $W_{V \times N}$ (V-dim to N-dim)
- N-dim hidden representation

**Output layer:**
- $W'_{N \times V}$ (N-dim to V-dim)
- V-dim output
- Predicts: **sat**

**We must learn W and W'**

**N will be the size of word vector**

# Semantic Properties of Embeddings

**The parallelogram model for analogy problems** (Rumelhart and Abrahamson, 1973): the location of *vine* can be found by subtracting *apple* from *tree* and adding *grape*.

```
       tree
        •
       / \
      /   \
apple•     •vine
      \   /
       \ /
        •
      grape
```

**Vector arithmetic:**
- *tree* - *apple* + *grape* ≈ *vine*

# Semantic Properties of Embeddings

**Relational properties of the GloVe vector space**, shown by projecting vectors onto two dimensions.

**(a)** *king* - *man* + *woman* is close to *queen*.
- Shows gender relationships: sister, aunt, niece on one side; brother, uncle, nephew on other side; with man, woman, heir, madam, sir, king in between

**(b)** offsets seem to capture comparative and superlative morphology.
- Shows gradation: slow → slower → slowest; short → shorter → shortest; strong → louder → strongest; clear/soft/dark → clearer/softer/darker → clearest/softest/darkest

*(Pennington et al., 2014)*

In [ ]:
# Demonstrate word embeddings using a simple corpus
from gensim.models import Word2Vec
import nltk

# Sample corpus
sentences = [
    ['king', 'is', 'a', 'strong', 'man'],
    ['queen', 'is', 'a', 'wise', 'woman'],
    ['boy', 'is', 'a', 'young', 'man'],
    ['girl', 'is', 'a', 'young', 'woman'],
    ['prince', 'is', 'a', 'young', 'king'],
    ['princess', 'is', 'a', 'young', 'queen'],
    ['man', 'is', 'strong'],
    ['woman', 'is', 'pretty'],
    ['prince', 'is', 'a', 'boy', 'will', 'be', 'king'],
    ['princess', 'is', 'a', 'girl', 'will', 'be', 'queen']
]

# Train Word2Vec model
model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, workers=4, epochs=100)

# Test semantic relationships
print("Word2Vec Semantic Properties")
print("="*60)

# Analogy: king - man + woman ≈ queen
try:
    result = model.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
    print("\nAnalogy: king - man + woman ≈ ?")
    for word, score in result:
        print(f"  {word}: {score:.3f}")
except:
    print("\nAnalogy: (need larger corpus for reliable analogies)")

# Similar words
try:
    print("\nWords similar to 'king':")
    for word, score in model.wv.most_similar('king', topn=3):
        print(f"  {word}: {score:.3f}")
except:
    print("\nSimilarity: (need larger corpus)")

## Summary & Key Takeaways

### **Distributional Hypothesis**
- Words in similar contexts have similar meanings
- Foundation for vector semantic models
- Firth (1957): "You shall know a word by the company it keeps"

### **Lexical Semantics**
- **Lemmas and senses**: Words have multiple meanings
- **Synonymy and similarity**: Related but distinct concepts
- **Semantic relations**: Frames, fields, connotation

### **Vector Representations**
- **Term-document matrices**: Documents as context
- **Term-term matrices**: Words as context
- **TF-IDF**: Weighing term importance
- **Cosine similarity**: Measuring semantic similarity

### **Word Embeddings (Word2Vec)**
- **Dense representations**: Low-dimensional vectors
- **CBOW**: Predict word from context (faster)
- **Skip-gram**: Predict context from word (better for rare words)
- **Semantic properties**: Analogies and relationships

### **Key Advantages**
- Learned from data (unsupervised)
- Capture semantic relationships
- Enable arithmetic operations on meaning
- Foundation for modern NLP (BERT, GPT, etc.)

### **Limitations**
- One vector per word (polysemy problem)
- Static representations (context-independent)
- Require large corpora for quality
- May encode societal biases

## Next Week Preview

**Topic**: Machine Learning in NLP  
**Focus**: Supervised learning, feature engineering, and classical ML approaches for NLP tasks

### Prepare for next week:
- Review basic machine learning concepts (classification, regression)
- Consider how vector representations enable ML algorithms
- Think about: What are the strengths and limitations of classical ML vs. deep learning?

---

## Discussion Questions

1. **Distributional semantics**: How does the distributional hypothesis relate to how humans learn word meanings?

2. **Vector representations**: What are the trade-offs between sparse (TF-IDF) and dense (Word2Vec) representations?

3. **Semantic properties**: Why can word embeddings capture analogies like "king - man + woman ≈ queen"?

4. **Polysemy problem**: How do static word embeddings handle words with multiple meanings?

5. **Bias in embeddings**: What societal biases might be encoded in word embeddings, and how can we address them?

6. **Future directions**: How do modern contextualized embeddings (BERT, ELMo) address limitations of Word2Vec?